In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121, EfficientNetB0
from tensorflow.keras.applications.densenet import preprocess_input as densenet_pre
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix

In [ ]:
data_dir = "/kaggle/input/datasets/afridirahman/brain-stroke-ct-image-dataset/Brain_Data_Organised"

img_size = (224,224)
batch_size = 16

In [ ]:
# DenseNet
train_gen_dense = ImageDataGenerator(
    preprocessing_function=densenet_pre,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

val_gen_dense = ImageDataGenerator(
    preprocessing_function=densenet_pre,
    validation_split=0.2
)

train_dense = train_gen_dense.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', subset='training'
)

val_dense = val_gen_dense.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', subset='validation', shuffle=False
)


# EfficientNet
train_gen_eff = ImageDataGenerator(
    preprocessing_function=eff_pre,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

val_gen_eff = ImageDataGenerator(
    preprocessing_function=eff_pre,
    validation_split=0.2
)

train_eff = train_gen_eff.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', subset='training'
)

val_eff = val_gen_eff.flow_from_directory(
    data_dir, target_size=img_size, batch_size=batch_size,
    class_mode='binary', subset='validation', shuffle=False
)

In [ ]:
base_dense = DenseNet121(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_dense.layers[:-30]:
    layer.trainable = False

x = base_dense.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

out_dense = Dense(1, activation='sigmoid')(x)

model_dense = Model(inputs=base_dense.input, outputs=out_dense)

model_dense.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

class_weights = {0:1.0, 1:2.5}

model_dense.fit(train_dense, validation_data=val_dense, epochs=10, class_weight=class_weights)

In [ ]:
base_eff = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))

for layer in base_eff.layers[:-30]:
    layer.trainable = False

x = base_eff.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

out_eff = Dense(1, activation='sigmoid')(x)

model_eff = Model(inputs=base_eff.input, outputs=out_eff)

model_eff.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_eff.fit(train_eff, validation_data=val_eff, epochs=10, class_weight=class_weights)

In [ ]:
val_dense.reset()
y_dense = model_dense.predict(val_dense)

val_eff.reset()
y_eff = model_eff.predict(val_eff)

y_true = val_dense.classes

In [ ]:
X_stack = np.hstack((y_dense, y_eff))

In [ ]:
meta_model = LogisticRegression()
meta_model.fit(X_stack, y_true)

In [ ]:
y_final_prob = meta_model.predict_proba(X_stack)[:,1]

In [ ]:
best_f1 = 0
best_t = 0.5

for t in np.arange(0.3, 0.6, 0.02):
    y_pred = (y_final_prob > t).astype(int)
    f1 = f1_score(y_true, y_pred)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best Threshold:", best_t)

In [ ]:
y_pred = (y_final_prob > best_t).astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.show()